# 03. Матанализ: производные, градиенты, цепное правило

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/03_calculus.ipynb)

Код из раздела [README.md](https://github.com/justxor/math-for-ai-2026/blob/main/README.md#m03). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### Численная проверка градиента

In [ ]:
import numpy as np

def grad_check(f, x, analytic, eps=1e-5):
    num = np.zeros_like(x)
    for i in range(x.size):
        e = np.zeros_like(x); e.flat[i] = eps
        num.flat[i] = (f(x + e) - f(x - e)) / (2 * eps)   # центральная разность, ошибка O(eps²)
    rel = np.linalg.norm(num - analytic) / (np.linalg.norm(num) + np.linalg.norm(analytic))
    return rel   # < 1e-7 — отлично, > 1e-4 — ищи баг

z = np.random.randn(5); y = np.eye(5)[2]
softmax = lambda z: np.exp(z - z.max()) / np.exp(z - z.max()).sum()
loss = lambda z: -np.log(softmax(z)[2])
print(grad_check(loss, z, softmax(z) - y))   # ~1e-10

### 🏋️ Практика модуля

**3.3.** Проверьте численно формулу $\partial L/\partial W = X^\top G$ для $L = \operatorname{sum}(XW)^2$.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
X = np.random.randn(4, 3); W = np.random.randn(3, 2)
f = lambda W: ((X @ W) ** 2).sum()
G = 2 * (X @ W)            # dL/dY
print(grad_check(f, W, X.T @ G))   # ~1e-10

## 🔬 Лаборатория: иллюстрации модуля

Это код картинок из курса. Меняйте параметры (learning rate, температуру, число точек, ядро…) и перезапускайте ячейку — так интуиция появляется быстрее всего.

In [ ]:
# Помощники для иллюстраций (те же, что в scripts/make_figures.py)
import matplotlib.pyplot as plt
from scipy import stats
C = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def save(fig, name):          # в ноутбуке просто показываем картинку
    fig.tight_layout(); plt.show()

def arrow(ax, v, color, label=None, origin=(0, 0), **kw):
    ax.annotate("", xy=(origin[0] + v[0], origin[1] + v[1]), xytext=origin,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, **kw))
    if label:
        ax.text(origin[0] + v[0] * 1.08, origin[1] + v[1] * 1.08, label, color=color, fontsize=13, fontweight="bold")

def rosen_like(x, y):
    return 0.5 * x**2 + 4 * y**2

In [ ]:
def taylor():
    x = np.linspace(-2 * np.pi, 2 * np.pi, 400)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(x, np.sin(x), color="k", lw=2.5, label="sin x")
    from math import factorial
    approx = np.zeros_like(x)
    for n, c in zip(range(0, 8), C * 2):
        k = 2 * n + 1
        approx = approx + (-1) ** n * x**k / factorial(k)
        if n in (0, 1, 2, 4):
            axes[0].plot(x, approx, color=c, lw=1.6, ls="--", label=f"до x^{k}")
    axes[0].set_ylim(-2, 2); axes[0].legend(fontsize=8); axes[0].set_title("Тейлор: чем больше членов, тем шире область точности")
    f = lambda x: np.log(1 + np.exp(x))
    x = np.linspace(-4, 4, 400); x0 = 1.0
    s = 1 / (1 + np.exp(-x0))
    axes[1].plot(x, f(x), color="k", lw=2.5, label="softplus(x)")
    axes[1].plot(x, f(x0) + s * (x - x0), color=C[0], ls="--", lw=1.8, label="1-й порядок (градиент)")
    axes[1].plot(x, f(x0) + s * (x - x0) + 0.5 * s * (1 - s) * (x - x0) ** 2, color=C[1], ls="--", lw=1.8, label="2-й порядок (гессиан)")
    axes[1].plot(x0, f(x0), "ko"); axes[1].set_ylim(-1, 5); axes[1].legend(fontsize=9)
    axes[1].set_title("GD использует 1-й порядок, метод Ньютона — 2-й")
    save(fig, "taylor")

taylor()